In [1]:
import duckdb
import pandas as pd

In [2]:
# Open a DuckDB connection. With no file name, the database lives only in memory:
# nothing is saved to disk, which is all we need to explore the data.
con = duckdb.connect()

> **Why did two new files appear in the project folder?**
>
> - `medallion_nobel.duckdb` is the database itself. `duckdb.connect('medallion_nobel.duckdb')` creates this file if it does not exist, and the `bronze`, `silver` and `gold` tables are stored in it.
> - `medallion_nobel.duckdb.wal` is the *write-ahead log*, a temporary helper file. Every change is written there first and only then merged into the main file, so DuckDB can recover your data if the program stops in the middle of a write.
>
> The `.wal` file disappears by itself when you run `con.close()` or shut down the kernel. Do not delete it by hand while the notebook is open: it may hold changes that are not in the `.duckdb` file yet.


## Exploring the raw JSON

Before building any layer, let's look at what the API gave us. DuckDB can query the JSON files directly, without loading them into a table first.


In [3]:
# What is at the top level of each file?
con.sql("DESCRIBE FROM '../data/bronze/*.json'").df()

,column_name,column_type,null,key,default,extra
0,laureates,"STRUCT(id VARCHAR, knownName STRUCT(en VARCHAR...",YES,None,None,None
1,meta,"STRUCT(""offset"" BIGINT, ""limit"" BIGINT, count ...",YES,None,None,None
2,links,"STRUCT(""first"" VARCHAR, self VARCHAR, ""next"" V...",YES,None,None,None


Each file has only three fields: `laureates` (a list with the actual data), `meta` (paging information) and `links`. One file is one page of the API.


### How many laureates per file?

In [4]:
con.sql("""
    SELECT len(laureates) AS laureates_in_page,
           meta.count     AS total_in_api
    FROM '../data/bronze/*.json'
""").df()


,laureates_in_page,total_in_api
0,100,1028
1,100,1028
2,100,1028
3,100,1028
4,100,1028
5,100,1028
6,100,1028
7,100,1028
8,100,1028
9,100,1028


### One row per laureate

In [5]:
# unnest opens the list: one row per laureate, one column per field
con.sql("""
    CREATE OR REPLACE TEMP VIEW laureates AS
    SELECT unnest(laureates, max_depth := 2)
    FROM '../data/bronze/*.json'
""")

con.sql("DESCRIBE laureates").df()


,column_name,column_type,null,key,default,extra
0,id,VARCHAR,YES,None,None,None
1,knownName,"STRUCT(en VARCHAR, se VARCHAR, ""no"" VARCHAR)",YES,None,None,None
2,givenName,"STRUCT(en VARCHAR, se VARCHAR, ""no"" VARCHAR)",YES,None,None,None
3,familyName,"STRUCT(en VARCHAR, se VARCHAR, ""no"" VARCHAR)",YES,None,None,None
4,fullName,"STRUCT(en VARCHAR, se VARCHAR, ""no"" VARCHAR)",YES,None,None,None
5,fileName,VARCHAR,YES,None,None,None
6,gender,VARCHAR,YES,None,None,None
7,birth,"STRUCT(date VARCHAR, ""year"" VARCHAR, place STR...",YES,None,None,None
8,wikipedia,"STRUCT(slug VARCHAR, english VARCHAR)",YES,None,None,None
9,wikidata,"STRUCT(id VARCHAR, url VARCHAR)",YES,None,None,None


### A first look

In [6]:
con.sql("""
    SELECT id, knownName, gender, birth.date, len(nobelPrizes) AS prizes
    FROM laureates
    LIMIT 5
""").df()


,id,knownName,gender,date,prizes
0,745,"{'en': 'A. Michael Spence', 'se': 'A. Michael ...",male,1943-11-07,1
1,102,"{'en': 'Aage N. Bohr', 'se': 'Aage N. Bohr', '...",male,1922-06-19,1
2,779,"{'en': 'Aaron Ciechanover', 'se': 'Aaron Ciech...",male,1947-10-01,1
3,259,"{'en': 'Aaron Klug', 'se': 'Aaron Klug', 'no':...",male,1926-08-11,1
4,1004,"{'en': 'Abdulrazak Gurnah', 'se': 'Abdulrazak ...",male,1948-00-00,1


### People and organizations are mixed
organizations does not have gender, so we can make this filter.

In [7]:
con.sql("""
    SELECT gender, count(*) AS total
    FROM laureates
    GROUP BY gender
""").df()


,gender,total
0,female,69
1,NaN,28
2,male,931


In [8]:
# The ones without gender are organizations
con.sql("""
    SELECT orgName.en AS organization, gender, birth
    FROM laureates
    WHERE gender IS NULL
    LIMIT 5
""").df()


,organization,gender,birth
0,American Friends Service Committee,None,<NA>
1,Amnesty International,None,<NA>
2,Center for Civil Liberties,None,<NA>
3,Doctors Without Borders,None,<NA>
4,European Union,None,<NA>


### Incomplete birth dates

In [9]:
con.sql("""
    SELECT knownName.en AS name, birth.date AS birth_date
    FROM laureates
    WHERE birth.date LIKE '%-00-00'
""").df()


,name,birth_date
0,Abdulrazak Gurnah,1948-00-00
1,Albert Lutuli,1898-00-00
2,Aleksey Yekimov,1945-00-00
3,Ardem Patapoutian,1967-00-00
4,David Baker,1962-00-00
5,David Card,1956-00-00
6,Gary Ruvkun,1952-00-00
7,James A. Robinson,1960-00-00
8,John Clarke,1942-00-00
9,John Jumper,1985-00-00


### More than one prize

In [10]:
con.sql("""
    SELECT knownName.en AS person, orgName.en AS organization, len(nobelPrizes) AS prizes
    FROM laureates
    WHERE len(nobelPrizes) > 1
    ORDER BY prizes DESC
""").df()


,person,organization,prizes
0,NaN,International Committee of the Red Cross,3
1,NaN,Office of the United Nations High Commissioner...,2
2,K. Barry Sharpless,NaN,2
3,Linus Pauling,NaN,2
4,Marie Curie,NaN,2
5,Frederick Sanger,NaN,2
6,John Bardeen,NaN,2


### Prizes by category

In [11]:
# nobelPrizes is also a list, so we unnest it too
con.sql("""
    SELECT prize.category.en    AS category,
           count(*)             AS prizes,
           min(prize.awardYear) AS first_year,
           max(prize.awardYear) AS last_year
    FROM (SELECT unnest(nobelPrizes) AS prize FROM laureates)
    GROUP BY category
    ORDER BY prizes DESC
""").df()


,category,prizes,first_year,last_year
0,Physiology or Medicine,235,1901,2026
1,Physics,231,1901,2026
2,Chemistry,202,1901,2026
3,Peace,144,1901,2026
4,Literature,123,1901,2026
5,Economic Sciences,101,1969,2026


In [12]:
con.close()